In [1]:
#generování dat
import numpy as np
import pandas as pd
from numpy.random import randn
N = 1000 # počet datových bodů
np.random.seed(42)
# pomocí gausovského rozdělení nagenerujeme body v prostoru váha-výška
# generování váhy v kg
def generuj_vahu(vaha_prumer = 80, vaha_sigma = 12, kolik = 100):
  vaha = vaha_prumer+randn(kolik)*vaha_sigma
  vaha[vaha < vaha_prumer-4*vaha_sigma] = vaha_prumer-4*vaha_sigma # orezani nepravdepodobnych hodnot - podvaha
  vaha[vaha > vaha_prumer+4*vaha_sigma] = vaha_prumer+4*vaha_sigma # orezani nepravdepodobnych hodnot - nadpodvaha
  return vaha

#generování výšky v cm
def generuj_vysku(vyska_prumer = 180, vyska_sigma = 15, kolik = 100):
  vyska = vyska_prumer+randn(kolik)*vyska_sigma
  vyska[vyska < vyska_prumer-4*vyska_sigma] = vyska_prumer-4*vyska_sigma
  vyska[vyska > vyska_prumer+4*vyska_sigma] = vyska_prumer+4*vyska_sigma
  return vyska

# spocteni body mass indexu BMI
vaha = generuj_vahu(kolik=N)
vyska = generuj_vysku(kolik=N)
bmi = vaha/(vyska/100)**2
data = {"vyska": vyska ,"vaha" : vaha, "bmi" : bmi}
df = pd.DataFrame(data) #
df.to_csv('data_lide.csv', index = False)
df.head(10)

,vyska,vaha,bmi
0,200.990332,85.960570,21.278889
1,193.869505,78.340828,20.843428
2,180.894456,87.772262,26.822965
3,170.295948,98.276358,33.887570
4,190.473350,77.190160,21.276172
5,185.902281,77.190357,22.335394
6,193.427898,98.950554,26.447218
7,189.527577,89.209217,24.835042
8,195.743291,74.366307,19.408968
9,171.971472,86.510721,29.252104


In [2]:
%matplotlib inline
promichano_index = df.index.to_list()
np.random.shuffle(promichano_index)

trenovaci_data_velikost = int(len(df)*0.80) # vezmeme 80 % pro nauceni modelu
print(df.index[10])
trenovaci_data = df.filter(promichano_index[:trenovaci_data_velikost], axis = 0) # vem nahodne indexy
testovaci_data = df.filter(promichano_index[trenovaci_data_velikost:], axis = 0) # vem nahodne indexy


10


In [3]:
import numpy.linalg as la

%matplotlib inline

#priprava dat pro linearni regresi
y = trenovaci_data['bmi']
X_t = np.array((np.ones(y.shape), trenovaci_data['vyska'], trenovaci_data['vaha']))
X = X_t.transpose()

# sestaveni matice a prave strany
A = X_t @ X # np.dot(X^T,X)
b = X_t @ y
# vypocet koeficientu resenim soustavy lin. rovnic
koeficienty = la.solve(A,b)
bmi_hat_trenovaci = X @ koeficienty # vypocet predikce na trenovacich datech


X_test_t = np.array(( np.ones(len(testovaci_data)),testovaci_data['vyska'], testovaci_data['vaha']))
X_test = X_test_t.transpose()
bmi_hat_testovaci = X_test @ koeficienty

# vypocet chyby
mse_ls_modelu_trenovaci = ((trenovaci_data['bmi']-bmi_hat_trenovaci)**2).mean()
mse_ls_modelu_testovaci = ((testovaci_data['bmi']-bmi_hat_testovaci)**2).mean()

print(f"Chyba na trenovacich datech puvodni{mse_ls_modelu_trenovaci}")
print(f"Chyba na testovacich datech puvodni {mse_ls_modelu_testovaci}")


Chyba na trenovacich datech puvodni0.9465364483745283
Chyba na testovacich datech puvodni 0.5918036668267495


In [4]:
X_t = np.array((np.ones(y.shape), trenovaci_data['vyska'], trenovaci_data['vaha'], trenovaci_data['vyska']**2, trenovaci_data['vaha']**2, trenovaci_data['vyska']*trenovaci_data['vaha']))
X = X_t.transpose()
# sestaveni matice a prave strany
A = X_t @ X # np.dot(X^T,X)
b = X_t @ y
# vypocet koeficientu resenim soustavy lin. rovnic
koeficienty = la.solve(A,b)
bmi_hat_trenovaci = X @ koeficienty # vypocet predikce na trenovacich datech

X_test_t = np.array((  np.ones(len(testovaci_data)),testovaci_data['vyska'], testovaci_data['vaha'], testovaci_data['vyska']**2, testovaci_data['vaha']**2, testovaci_data['vyska']*testovaci_data['vaha']))

X_test = X_test_t.transpose()
bmi_hat_testovaci = X_test @ koeficienty

# vypocet chyby
mse_ls_modelu_trenovaci = ((trenovaci_data['bmi']-bmi_hat_trenovaci)**2).mean()
mse_ls_modelu_testovaci = ((testovaci_data['bmi']-bmi_hat_testovaci)**2).mean()

print(f"Chyba na trenovacich datech nova {mse_ls_modelu_trenovaci}")
print(f"Chyba na testovacich datech nova {mse_ls_modelu_testovaci}")

Chyba na trenovacich datech nova 0.02488274048047639
Chyba na testovacich datech nova 0.015044218282599626
